In [1]:
!pip install "kaleido<1.0"

In [2]:
input_path = '/mmdetection3d/data/indoor_spray/scenes_full'
output_path = '/mmdetection3d/plots_and_tables'

## Intensity and points quantity plots

In [3]:

import os

import pandas as pd
import numpy as np
from tqdm import tqdm




CLASSES_MAP = {
    'target': 1,
    'car': 2,
    'spray': 3
}



def generate_tables_data(scenes_folders):
    scenes_files = [os.listdir(os.path.join(folder, 'points')) for folder in scenes_folders]
    folder_names = [os.path.basename(folder) for folder in scenes_folders]
    pbar = tqdm(total=sum(len(files) for files in scenes_files), desc="Processing frames")
    data = {}

    scene_target_points_quantity = {
        '01_target_dry_10m': 55,
        '02_target_spray-1_10m': 55,
        '03_target_spray-2_10m': 55,
        '04_target_dry_20m': 18,
        '05_target_spray-1_20m': 18,
        '06_target_spray-2_20m': 18,
        '07_target_dry_30m': 8,
        '08_target_spray-1_30m': 8,
        '09_target_spray-2_30m': 8
    }
    
    scenes_map = {
        '01_target_dry_10m': '01_target_dry_10m',
        '02_target_spray-1_10m': '01_target_dry_10m',
        '03_target_spray-2_10m': '01_target_dry_10m',
        '04_target_dry_20m': '04_target_dry_20m',
        '05_target_spray-1_20m': '04_target_dry_20m',
        '06_target_spray-2_20m': '04_target_dry_20m',
        '07_target_dry_30m': '07_target_dry_30m',
        '08_target_spray-1_30m': '07_target_dry_30m',
        '09_target_spray-2_30m': '07_target_dry_30m',
        '10_scenario-a_dry_10m': '10_scenario-a_dry_10m',
        '11_scenario-a_spray-2_10m': '10_scenario-a_dry_10m',
        '12_scenario-a_dry_20m': '12_scenario-a_dry_20m',
        '13_scenario-a_spray-2_20m': '12_scenario-a_dry_20m',
        '14_scenario-a_dry_30m': '14_scenario-a_dry_30m',
        '15_scenario-a_spray-2_30m': '14_scenario-a_dry_30m',
        '16_scenario-b_dry_10m': '16_scenario-b_dry_10m',
        '17_scenario-b_spray-2_10m': '16_scenario-b_dry_10m',
        '18_scenario-b_dry_20m': '18_scenario-b_dry_20m',
        '19_scenario-b_spray-2_20m': '18_scenario-b_dry_20m',
        '20_scenario-b_dry_30m': '20_scenario-b_dry_30m',
        '21_scenario-b_spray-2_30m': '20_scenario-b_dry_30m',
        '22_scenario-c_dry_5m': '22_scenario-c_dry_5m',
        '23_scenario-c_spray-2_5m': '22_scenario-c_dry_5m',
        '24_scenario-c_dry_10m': '24_scenario-c_dry_10m',
        '25_scenario-c_spray-2_10m': '24_scenario-c_dry_10m'
    }

    def points_quantity_transform(data):
        return data

    def intensity_transform(data):
        return np.concatenate(data).flatten()
    
    transform_functions = {
        'points_quantity': points_quantity_transform,
        'intensity': intensity_transform,
        'intensity_lost_as_zeros': intensity_transform
    }

    for folder_path, folder_name, files in zip(scenes_folders, folder_names, scenes_files):
        data[folder_name] = {
            class_name: {
                'intensity': [],
                'points_quantity': []
            } for class_name in CLASSES_MAP.keys()
        }
        for file in files:
            points = np.fromfile(os.path.join(folder_path, 'points', file), dtype=np.float32).reshape(-1, 4)
            labels = np.fromfile(os.path.join(folder_path, 'full_labels', file), dtype=np.uint8)
            for class_name, class_id in CLASSES_MAP.items():
                class_points = points[labels == class_id]
                if class_points.size > 0:
                    data[folder_name][class_name]['intensity'].append(class_points[:, 3])
                    data[folder_name][class_name]['points_quantity'].append(class_points.shape[0])
            pbar.update(1)
    
    pbar.close()

    print('Calculating statistics...')
    
    for folder_name in folder_names:
        data[folder_name]['target']['intensity_lost_as_zeros'] = []
        if folder_name in scene_target_points_quantity:
            for intensity_arr in data[folder_name]['target']['intensity']:
                intensity_with_zeros = np.zeros(scene_target_points_quantity[folder_name])
                intensity_with_zeros[:len(intensity_arr)] = intensity_arr
                data[folder_name]['target']['intensity_lost_as_zeros'].append(intensity_with_zeros)

    dict_for_empty_classes = {
        'mean': '-',
        'std': '-',
        'min': '-',
        '25%': '-',
        '50%': '-',
        '75%': '-',
        'max': '-'
    }

    for folder_name in folder_names:
        for class_name in CLASSES_MAP.keys():
            for stat_key in data[folder_name][class_name].keys():
                if not data[folder_name][class_name][stat_key]:
                    data[folder_name][class_name][stat_key] = dict_for_empty_classes
                else:
                    transform_func = transform_functions[stat_key]
                    describe_data = pd.DataFrame(
                        transform_func(data[folder_name][class_name][stat_key])
                    ).describe()

                    data[folder_name][class_name][stat_key] = {
                        'mean': describe_data.loc['mean'][0].round(2),
                        'std': describe_data.loc['std'][0].round(2),
                        'min': describe_data.loc['min'][0].round(2),
                        '25%': describe_data.loc['25%'][0].round(2),
                        '50%': describe_data.loc['50%'][0].round(2),
                        '75%': describe_data.loc['75%'][0].round(2),
                        'max': describe_data.loc['max'][0].round(2),
                        'original_mean': describe_data.loc['mean'][0]
                    }
    dfs = {}
    for class_name in CLASSES_MAP.keys():
        dfs[class_name] = {}
        for stat_key in data[folder_names[0]][class_name].keys():
            dfs[class_name][stat_key] = []
            for folder_name in folder_names:
                result_dict = {
                    'scene': folder_name,
                    **data[folder_name][class_name][stat_key]
                }
                if result_dict['mean'] != '-' and data[scenes_map[folder_name]][class_name][stat_key]['mean'] != '-':
                    base_mean = data[scenes_map[folder_name]][class_name][stat_key]['original_mean']
                    current_mean = result_dict['original_mean']
                    increase_percentage = (current_mean - base_mean) / base_mean * 100 if base_mean != 0 else 0
                    result_dict['increase_percentage'] = round(increase_percentage, 2)
                else:
                    result_dict['increase_percentage'] = '-'
                
                if 'original_mean' in result_dict:
                    del result_dict['original_mean']
                
                dfs[class_name][stat_key].append(result_dict)
                
            dfs[class_name][stat_key] = pd.DataFrame(dfs[class_name][stat_key])
        
    return dfs

def save_as_csv(dfs, output_path):
    final_output_path = os.path.join(output_path, 'tables_csv')
    os.makedirs(final_output_path, exist_ok=True)
    for class_name, tables in dfs.items():
        for data_type, df in tables.items():
            output_file = os.path.join(final_output_path, f"{class_name}_{data_type}.csv")
            df.to_csv(output_file, index=False)


os.makedirs(output_path, exist_ok=True)

scene_folders = []
folders_to_use = [folder for folder in os.listdir(input_path) if os.path.isdir(os.path.join(input_path, folder))]
for folder in folders_to_use:
    scene_folders.append(os.path.join(input_path, folder))

scene_folders.sort()

dfs = generate_tables_data(scene_folders)

print('Saving tables as CSV files...')
save_as_csv(dfs, output_path)

Processing frames: 100%|██████████| 2216/2216 [00:01<00:00, 1171.52it/s]


Calculating statistics...
Saving tables as CSV files...


In [4]:
import pandas as pd
import os
import plotly.express as px

In [5]:
def load_results_table(file_path):
    SPRAY_NAME_MAP = {
        'dry': 'dry',
        'spray-1': 'intensity 1',
        'spray-2': 'intensity 2'
    }

    df = pd.read_csv(file_path)

    df['scene_type'] = df['scene'].apply(lambda x: x.split('_')[1].replace('-', ' '))
    df['spray'] = df['scene'].apply(lambda x: SPRAY_NAME_MAP[x.split('_')[2]])
    df['distance'] = df['scene'].apply(lambda x: x.split('_')[3])
    
    return df

In [6]:
car_intensity_df = load_results_table(os.path.join(output_path, 'tables_csv', 'car_intensity.csv'))
car_points_quantity_df = load_results_table(os.path.join(output_path, 'tables_csv', 'car_points_quantity.csv'))
spray_intensity_df = load_results_table(os.path.join(output_path, 'tables_csv', 'spray_intensity.csv'))
spray_points_quantity_df = load_results_table(os.path.join(output_path, 'tables_csv', 'spray_points_quantity.csv'))
target_intensity_df = load_results_table(os.path.join(output_path, 'tables_csv', 'target_intensity.csv'))
target_points_quantity_df = load_results_table(os.path.join(output_path, 'tables_csv', 'target_points_quantity.csv'))

In [7]:
car_points_quantity_df.iloc[:9] = target_points_quantity_df.iloc[:9]
car_points_quantity_df = car_points_quantity_df[car_points_quantity_df['spray'] != 'dry']
car_points_quantity_df['decrease_percentage'] = -car_points_quantity_df['increase_percentage'].astype(float).round(2)
car_points_quantity_df['scene_type'] = car_points_quantity_df['scene_type'].replace({
    'target': 'Lambertian target',
    'scenario a': 'Scenario A',
    'scenario b': 'Scenario B',
    'scenario c': 'Scenario C'
})

car_points_quantity_df = car_points_quantity_df[car_points_quantity_df['spray'] != 'intensity 1']

In [8]:
car_intensity_df.iloc[:9] = target_intensity_df.iloc[:9]
car_intensity_df = car_intensity_df[car_intensity_df['spray'] != 'dry']
car_intensity_df['decrease_percentage'] = -car_intensity_df['increase_percentage'].astype(float).round(2)
car_intensity_df['scene_type'] = car_intensity_df['scene_type'].replace({
    'target': 'Lambertian target',
    'scenario a': 'Scenario A',
    'scenario b': 'Scenario B',
    'scenario c': 'Scenario C'
})
car_intensity_df["text"] = (
    car_intensity_df["decrease_percentage"]
    .map(lambda x: f"{x:.1f}")
)

car_intensity_df = car_intensity_df[car_intensity_df['spray'] != 'intensity 1']

In [9]:

def create_bar_plot(df, x, y, output_path, text='decrease_percentage', color='distance' , width=750, height=350, show_negative=False):
    fig = px.bar(
        df,
        x=x,
        y=y,
        color=color,
        barmode='group',
        text=text,
        category_orders = {
            "distance": ["5m", "10m", "20m", "30m"],
            "scene_type": ["Lambertian target", "Scenario A", "Scenario B", "Scenario C"]
        },
        color_discrete_map={
            "5m": "#FE6100",
            "10m": "#648FFF",
            "20m": "#DC267F",
            "30m": "#FFB000"
        },
        labels = {
            "decrease_percentage": "Decrease percentage (%)",
        }
    )
    
    fig.update_traces(textposition="outside")
    fig.update_xaxes(title=None)
    
    fig.update_layout(
        width=width,
        height=height
    )
    
    max_y = df[y].max()
    fig.update_yaxes(range=[0, max_y * 1.15])
    if show_negative:
        min_y = df[y].min()
        fig.update_yaxes(range=[min_y - 3, max_y * 1.15])

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    fig.write_image(output_path + '.png')
    fig.write_image(output_path + '.svg')
    
    return fig

In [10]:

# All
create_bar_plot(
    car_points_quantity_df,
    x='scene_type',
    y='decrease_percentage',
    output_path=os.path.join(output_path, 'points_quantity', 'all')
)

/tmp/ipykernel_184638/3944924256.py:40: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/tmp/ipykernel_184638/3944924256.py:41: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




In [11]:

# All
create_bar_plot(
    car_intensity_df,
    x='scene_type',
    y='decrease_percentage',
    output_path=os.path.join(output_path, 'intensity', 'all'),
    show_negative=True
)

/tmp/ipykernel_184638/3944924256.py:40: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/tmp/ipykernel_184638/3944924256.py:41: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Generate AP LaTeX tables

In [14]:
import numpy as np

def generate_multi_algo_ap_loss_latex_table(algo_dict):
    """
    algo_dict: {
        "Algo1": df1,
        "Algo2": df2,
        ...
    }
    Each df must contain: ['scene', 'AP_05_dry', 'AP_05_spray']
    """

    # Prepare processed data per algorithm
    processed = {}
    for algo_name, df in algo_dict.items():
        temp = df[['scene', 'AP_05_dry', 'AP_05_spray']].copy()
        temp.columns = ['Scenario', 'AP Dry', 'AP Spr']
        temp[['AP Dry', 'AP Spr']] = temp[['AP Dry', 'AP Spr']].round(3)

        temp['AP Loss (\%)'] = (
            -(df['AP_05_spray'] - df['AP_05_dry']) / df['AP_05_dry'] * 100
        ).round(2)
        # replace nan, inf and -inf with "N/A"
        temp['AP Loss (\%)'] = temp['AP Loss (\%)'].replace([np.nan, np.inf, -np.inf], "N/A")

        processed[algo_name] = temp

    # Assume all scenarios are identical → take from first df
    scenarios = list(processed.values())[0]['Scenario']

    # Number of algorithms
    n_algos = len(processed)

    # Column format: 1 (Scenario) + 3 per algo
    col_format = "l" + "ccc" * n_algos

    latex_str = "\\begin{table*}\n\\resizebox{\\textwidth}{!}{\n"
    latex_str += f"\\begin{{tabular}}{{{col_format}}}\n"
    latex_str += "\\toprule\n"

    # ---- First header row (algorithms) ----
    header_row_1 = [""]

    for algo_name in processed.keys():
        header_row_1.append(f"\\multicolumn{{3}}{{c}}{{{algo_name}}}")

    latex_str += " & ".join(header_row_1) + " \\\\\n"

    # ---- cmidrules ----
    start = 2
    cmidrules = []
    for i in range(n_algos):
        cmidrules.append(f"\\cmidrule(lr){{{start}-{start+2}}}")
        start += 3
    latex_str += " ".join(cmidrules) + "\n"

    # ---- Second header row ----
    header_row_2 = ["Scenario"]
    for _ in processed.keys():
        header_row_2 += ["AP Dry", "AP Spr", "AP Loss (\%)"]

    latex_str += " & ".join(header_row_2) + " \\\\\n"
    latex_str += "\\midrule\n"

    # ---- Body rows ----
    num_rows = len(scenarios)

    for i in range(num_rows):
        row = [str(scenarios.iloc[i])]

        for algo_name in processed.keys():
            df_algo = processed[algo_name]
            row += [
                str(df_algo.iloc[i]['AP Dry']),
                str(df_algo.iloc[i]['AP Spr']),
                str(df_algo.iloc[i]['AP Loss (\%)']),
            ]

        latex_str += " & ".join(row) + " \\\\\n"

        # midrule before summary rows (same logic as yours)
        if i == 7:
            latex_str += "\\midrule\n"

    latex_str += "\\bottomrule\n"
    latex_str += "\\end{tabular}}\n"
    latex_str += "\\caption{AP comparison between dry and spray scenarios for multiple algorithms.}\n"
    latex_str += "\\label{tab:ap_loss_comparison_multi}\n"
    latex_str += "\\end{table*}"

    return latex_str

latex_str = generate_multi_algo_ap_loss_latex_table({
    "BEVFusion LiDAR": pd.read_csv('/mmdetection3d/results/bevfusion_lidar/evaluation_results.csv'),
    'CenterPoint': pd.read_csv('/mmdetection3d/results/centerpoint/evaluation_results.csv'),
    'PointPillars': pd.read_csv('/mmdetection3d/results/pointpillars/evaluation_results.csv'),
    'SSN': pd.read_csv('/mmdetection3d/results/ssn/evaluation_results.csv')
})

print(latex_str)

with open(os.path.join(output_path, 'ap_loss_comparison_multi.tex'), 'w') as f:
    f.write(latex_str)

\begin{table*}
\resizebox{\textwidth}{!}{
\begin{tabular}{lcccccccccccc}
\toprule
 & \multicolumn{3}{c}{BEVFusion LiDAR} & \multicolumn{3}{c}{CenterPoint} & \multicolumn{3}{c}{PointPillars} & \multicolumn{3}{c}{SSN} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}
Scenario & AP Dry & AP Spr & AP Loss (\%) & AP Dry & AP Spr & AP Loss (\%) & AP Dry & AP Spr & AP Loss (\%) & AP Dry & AP Spr & AP Loss (\%) \\
\midrule
A 10m & 0.968 & 0.166 & 82.85 & 0.92 & 0.124 & 86.55 & 0.0 & 0.0 & N/A & 0.16 & 0.001 & 99.42 \\
A 20m & 1.0 & 0.612 & 38.79 & 0.946 & 0.543 & 42.67 & 0.0 & 0.0 & N/A & 0.201 & 0.085 & 57.86 \\
A 30m & 0.995 & 0.815 & 18.11 & 1.0 & 0.681 & 31.89 & 0.01 & 0.034 & -260.69 & 0.034 & 0.002 & 92.72 \\
B 10m & 0.953 & 0.931 & 2.32 & 1.0 & 0.65 & 34.95 & 0.995 & 0.932 & 6.32 & 0.382 & 0.268 & 29.95 \\
B 20m & 0.975 & 0.841 & 13.76 & 0.996 & 0.761 & 23.61 & 0.287 & 0.168 & 41.46 & 0.088 & 0.078 & 11.11 \\
B 30m & 0.999 & 0.847 & 15.29 & 1.0 & 0.365 & 